# Cell 1 — Mount Drive and install


In [ ]:
from pathlib import Path
import os, subprocess, sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

DRIVE_BASE = Path('/content/drive/MyDrive/Verdant-Minds')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/Verdant-Minds')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/example/Verdant-Minds.git', str(REPO_DIR)], check=False)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

req = REPO_DIR / 'requirements.txt'
if req.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=False)

os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('DRIVE_BASE =', DRIVE_BASE)
print('REPO_DIR =', REPO_DIR)


# Cell 2 — Load system from checkpoint


In [ ]:
from pathlib import Path
import json
from verdant.system import VerdantSystem
from verdant.io.query_interface import QueryInterface

CHECKPOINT_PATH = DRIVE_BASE / 'checkpoints' / 'verdant_latest.json'
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

system = VerdantSystem()
if CHECKPOINT_PATH.exists():
    system.load_state(str(CHECKPOINT_PATH))
    print('Loaded checkpoint:', CHECKPOINT_PATH)
else:
    print('No checkpoint found; starting fresh.')

query_interface = QueryInterface(system)

metrics = system.get_metrics()
BASELINE_STATS = {
    'node_count': int(system.memory_web.graph.number_of_nodes()),
    'edge_count': int(system.memory_web.graph.number_of_edges()),
    'basin_count': int(metrics.get('basin_count', 0)),
    't_g': float(metrics.get('t_g', 0.5)),
}
TG_TRAJECTORY = [BASELINE_STATS['t_g']]
TRAINING_TOP_ACTIVATIONS = {}
TRAINING_COVERAGE = {'expected': 0, 'hit': 0}
BASIN_SNAPSHOT_BEFORE = {str(b.get('basin_id')): len(b.get('nodes', [])) for b in metrics.get('basins', []) if isinstance(b, dict)}

print('Graph stats:')
print(json.dumps(BASELINE_STATS, indent=2))


# Cell 3 — Feed phoneme inventory


In [ ]:
from pathlib import Path
import json

phoneme_path = Path('corpus/phonemes/english_phoneme_inventory.json')
phonemes = json.loads(phoneme_path.read_text(encoding='utf-8'))

nodes_before = system.memory_web.graph.number_of_nodes()
edges_before = system.memory_web.graph.number_of_edges()

for i, entry in enumerate(phonemes, 1):
    symbol = entry.get('symbol_ipa', '')
    art = entry.get('articulation', {})
    desc = art.get('description', '')
    examples = ', '.join([x.get('word', '') for x in entry.get('example_words', [])[:4]])
    sentence = f"Phoneme {symbol} is described as {desc}. Example words include {examples}."
    system.process_input(sentence, metadata={
        'source': 'corpus_ingest',
        'corpus_file': str(phoneme_path),
        'entry_id': entry.get('phoneme_id', f'ph_{i:03d}')
    })

    node_label = f"phoneme:{symbol}"
    system.memory_web.add_concept(node_label, metadata={'phoneme_id': entry.get('phoneme_id'), 'schema_version': entry.get('schema_version')})
    for edge in entry.get('graph_edges', []):
        target = str(edge.get('to', '')).strip()
        if not target:
            continue
        if system.memory_web.get_concept(target) is None:
            system.memory_web.add_concept(target)
        system.memory_web.connect(node_label, target, weight=float(edge.get('weight', 0.5)))

    if i % 10 == 0:
        print(f"Processed {i}/{len(phonemes)} phonemes")

nodes_after = system.memory_web.graph.number_of_nodes()
edges_after = system.memory_web.graph.number_of_edges()
metrics = system.get_metrics(); TG_TRAJECTORY.append(float(metrics.get('t_g', 0.5)))
print('Phoneme ingest complete')
print('New nodes:', nodes_after - nodes_before)
print('New edges:', edges_after - edges_before)


# Cell 4 — Feed grammar rules


In [ ]:
from pathlib import Path
import json

grammar_path = Path('corpus/grammar/english_grammar_rules.json')
rules = json.loads(grammar_path.read_text(encoding='utf-8'))

unlock_chains = []
for i, rule in enumerate(rules, 1):
    rule_id = rule.get('rule_id', f'gr_{i:05d}')
    name = rule.get('rule_name', '')
    formal = rule.get('formal_statement', '')
    canonical = (rule.get('canonical_example') or {}).get('correct', '')
    text = f"Grammar rule {name}. {formal} Canonical example: {canonical}."
    system.process_input(text, metadata={'source':'corpus_ingest','corpus_file':str(grammar_path),'entry_id':rule_id})

    if system.memory_web.get_concept(rule_id) is None:
        system.memory_web.add_concept(rule_id, metadata={'type':'grammar_rule'})
    for edge in rule.get('graph_edges_this_rule_creates', []):
        src = str(edge.get('from', ''))
        dst = str(edge.get('to', ''))
        w = float(edge.get('weight', 0.6))
        if src and system.memory_web.get_concept(src) is None:
            system.memory_web.add_concept(src)
        if dst and system.memory_web.get_concept(dst) is None:
            system.memory_web.add_concept(dst)
        if src and dst:
            system.memory_web.connect(src, dst, weight=w)

    prereqs = [str(x) for x in rule.get('prerequisite_rules', [])]
    unlocks = [str(x) for x in rule.get('unlocks_rules', [])]
    for p in prereqs:
        if system.memory_web.get_concept(p) is None:
            system.memory_web.add_concept(p, metadata={'type':'grammar_rule'})
        if system.memory_web.get_concept(rule_id) is None:
            system.memory_web.add_concept(rule_id, metadata={'type':'grammar_rule'})
        system.memory_web.connect(p, rule_id, weight=0.9)
        unlock_chains.append((p, rule_id))
    for u in unlocks:
        if system.memory_web.get_concept(u) is None:
            system.memory_web.add_concept(u, metadata={'type':'grammar_rule'})
        system.memory_web.connect(rule_id, u, weight=0.9)
        unlock_chains.append((rule_id, u))

    if i % 10 == 0:
        print(f"Processed {i}/{len(rules)} grammar rules")

metrics = system.get_metrics(); TG_TRAJECTORY.append(float(metrics.get('t_g', 0.5)))
print('Grammar ingest complete. Unlock chains formed:', len(unlock_chains))
print('Sample unlock chains:', unlock_chains[:15])


# Cell 5 — Feed lexicon


In [ ]:
from pathlib import Path
import json

lex_path = Path('corpus/lexicon/core_vocabulary_physical_world.json')
lex = json.loads(lex_path.read_text(encoding='utf-8'))

nodes_before = system.memory_web.graph.number_of_nodes()
edges_before = system.memory_web.graph.number_of_edges()

for i, entry in enumerate(lex, 1):
    wid = entry.get('word_id', f'w_{i:05d}')
    word = entry.get('word', '')
    sem = entry.get('semantics', {})
    primary = sem.get('primary_definition', '')
    sense1 = (sem.get('definitions_by_sense') or [{}])[0]
    ex = sense1.get('example', '')
    coll = (((sem.get('collocations') or {}).get('strong_collocates')) or [])[:3]
    coll_text = ', '.join(coll)
    text = f"Word {word}. Definition: {primary}. Example: {ex}. Strong collocates: {coll_text}."
    chunk = system.process_input(text, metadata={'source':'corpus_ingest','corpus_file':str(lex_path),'entry_id':wid})

    seeded_edges = 0
    for edge in (entry.get('graph_seeding') or {}).get('suggested_edge_targets', []):
        target = str(edge.get('target', '')).strip()
        if not target:
            continue
        if system.memory_web.get_concept(word) is None:
            system.memory_web.add_concept(word)
        if system.memory_web.get_concept(target) is None:
            system.memory_web.add_concept(target)
        system.memory_web.connect(word, target, weight=float(edge.get('weight', 0.6)))
        seeded_edges += 1

    for ra in query_interface.recent_activations(10):
        c = ra.get('concept')
        if not c:
            continue
        TRAINING_TOP_ACTIVATIONS[c] = TRAINING_TOP_ACTIVATIONS.get(c, 0) + 1

    if i % 25 == 0:
        m = system.get_metrics(); TG_TRAJECTORY.append(float(m.get('t_g', 0.5)))
        print(f"Processed {i}/{len(lex)} words | new_nodes={system.memory_web.graph.number_of_nodes()-nodes_before} | new_edges={system.memory_web.graph.number_of_edges()-edges_before} | t_g={float(m.get('t_g',0.5)):.4f}")

m = system.get_metrics(); TG_TRAJECTORY.append(float(m.get('t_g', 0.5)))
print('Lexicon ingest complete')

basins = m.get('basins', [])
basin_sizes = {str(b.get('basin_id')): len(b.get('nodes', [])) for b in basins if isinstance(b, dict)}
delta = []
for bid,size in basin_sizes.items():
    prev = BASIN_SNAPSHOT_BEFORE.get(bid, 0)
    delta.append((bid, size-prev))
delta.sort(key=lambda x: x[1], reverse=True)
print('Basins with most new members:', delta[:10])


# Cell 6 — Feed teaching sentences


In [ ]:
from pathlib import Path
import json

sent_path = Path('corpus/sentences/teaching_sentences_physical_world.json')
sentences = json.loads(sent_path.read_text(encoding='utf-8'))

nodes_before = system.memory_web.graph.number_of_nodes()
edges_before = system.memory_web.graph.number_of_edges()
emergent_before = set(system.memory_web.get_emergent_nodes())
coverage_hits = 0
coverage_expected = 0

for i, entry in enumerate(sentences, 1):
    sid = entry.get('sentence_id', f'ts_{i:05d}')
    text = entry.get('sentence', '')
    chunk = system.process_input(text, metadata={'source':'corpus_ingest','corpus_file':str(sent_path),'entry_id':sid})

    for edge in ((entry.get('relational_map') or {}).get('graph_edges_this_sentence_reinforces') or []):
        src = str(edge.get('from', '')).strip(); dst = str(edge.get('to', '')).strip()
        if not src or not dst:
            continue
        if system.memory_web.get_concept(src) is None:
            system.memory_web.add_concept(src)
        if system.memory_web.get_concept(dst) is None:
            system.memory_web.add_concept(dst)
        system.memory_web.connect(src, dst, weight=float(edge.get('weight_delta', 0.1)))

    expected = ((entry.get('relational_map') or {}).get('concept_activations_expected') or [])
    recent = {x.get('concept') for x in query_interface.recent_activations(30) if x.get('concept')}
    hit = sum(1 for c in expected if c in recent)
    coverage_hits += hit
    coverage_expected += len(expected)
    TRAINING_COVERAGE['hit'] += hit
    TRAINING_COVERAGE['expected'] += len(expected)

    for ra in query_interface.recent_activations(20):
        c = ra.get('concept')
        if c:
            TRAINING_TOP_ACTIVATIONS[c] = TRAINING_TOP_ACTIVATIONS.get(c, 0) + 1

    if i % 10 == 0:
        m = system.get_metrics(); TG_TRAJECTORY.append(float(m.get('t_g', 0.5)))
        emergent_now = set(system.memory_web.get_emergent_nodes())
        emergent_batch = len(emergent_now - emergent_before)
        cov = (coverage_hits / max(1, coverage_expected))*100.0
        print(f"Processed {i}/{len(sentences)} sentences | coverage={cov:.2f}% | emergent_new={emergent_batch} | t_g={float(m.get('t_g',0.5)):.4f}")
        emergent_before = emergent_now

m = system.get_metrics(); TG_TRAJECTORY.append(float(m.get('t_g', 0.5)))
final_cov = (coverage_hits / max(1, coverage_expected))*100.0
print('Sentence ingest complete')
print('Total new concepts:', system.memory_web.graph.number_of_nodes()-nodes_before)
print('Total new edges:', system.memory_web.graph.number_of_edges()-edges_before)
print('Average activation coverage:', f"{final_cov:.2f}%")
print('Top 10 most activated concepts:', sorted(TRAINING_TOP_ACTIVATIONS.items(), key=lambda x: x[1], reverse=True)[:10])


# Cell 7 — Checkpoint save (atomic write)


In [ ]:
from pathlib import Path

checkpoint_dir = DRIVE_BASE / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'verdant_latest.json'
tmp_path = checkpoint_dir / 'verdant_latest.json.tmp'

system.save_state(str(tmp_path))
tmp_path.replace(checkpoint_path)

m = system.get_metrics()
print('Checkpoint saved:', checkpoint_path)
print({'node_count': system.memory_web.graph.number_of_nodes(), 'edge_count': system.memory_web.graph.number_of_edges(), 'basin_count': m.get('basin_count'), 't_g': m.get('t_g')})


# Cell 8 — See and Say session


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from verdant.io.see_and_say import run_session

result = run_session('corpus/lists/teaching_list_physical_world.json', query_interface)
rows = []
rows.append("<table border='1' cellpadding='6' cellspacing='0'>")
rows.append("<tr><th>Cloze ID</th><th>Masked Sentence</th><th>Correct</th><th>Pass/Fail</th><th>Δ(correct-best distractor)</th></tr>")
for item in result.get('results', []):
    cid = item.get('cloze_id', '')
    masked = item.get('masked_sentence', '')
    target = item.get('target', '')
    passed = bool(item.get('passed', False))
    color = '#d4edda' if passed else '#f8d7da'
    delta = float(item.get('target_score', 0.0)) - float(item.get('max_distractor_score', 0.0))
    rows.append(f"<tr style='background:{color}'><td>{cid}</td><td>{masked}</td><td>{target}</td><td>{'PASS' if passed else 'FAIL'}</td><td>{delta:.6f}</td></tr>")
rows.append("</table>")

overall = result.get('pass_rate', 0.0)
weak = ', '.join([f"{c}({n})" for c,n in result.get('weakest_concepts', [])]) or 'None'
summary = f"<p><b>Overall pass rate:</b> {overall:.2%}<br><b>Weakest concepts:</b> {weak}<br><b>Strongest causal chains:</b> from highest-evidence PASS items in table above.</p>"
display(HTML(summary + ''.join(rows)))


# Cell 9 — Graph inspection after training


In [ ]:
import json

metrics = system.get_metrics()
node_now = system.memory_web.graph.number_of_nodes()
edge_now = system.memory_web.graph.number_of_edges()
new_nodes = node_now - BASELINE_STATS['node_count']
new_edges = edge_now - BASELINE_STATS['edge_count']

basins_now = {str(b.get('basin_id')): len(b.get('nodes', [])) for b in metrics.get('basins', []) if isinstance(b, dict)}
growth = []
for bid, size in basins_now.items():
    growth.append((bid, size - BASIN_SNAPSHOT_BEFORE.get(bid, 0)))
growth.sort(key=lambda x: x[1], reverse=True)
new_basins = [bid for bid in basins_now if bid not in BASIN_SNAPSHOT_BEFORE]

top20 = sorted(TRAINING_TOP_ACTIVATIONS.items(), key=lambda x: x[1], reverse=True)[:20]
teaching_targets = json.loads(Path('corpus/lists/teaching_list_physical_world.json').read_text()).get('target_concepts', [])
weak_targets = []
for t in teaching_targets:
    if TRAINING_TOP_ACTIVATIONS.get(t, 0) < 2:
        weak_targets.append(t)

print('Before/After Summary')
print('New nodes added total:', new_nodes)
print('New edges added total:', new_edges)
print('Basins grew most:', growth[:10])
print('New basins formed:', new_basins)
print('Top 20 activated concepts:', top20)
print('Weak teaching targets (low activation history):', weak_targets)
print('T_g trajectory samples:', [round(x,4) for x in TG_TRAJECTORY])
print('T_g stability:', 'stable' if max(TG_TRAJECTORY)-min(TG_TRAJECTORY) < 0.25 else 'heated during run')


# Cell 10 — Live concept inspection


In [ ]:
concept = 'heavy'
antonym = 'light'

info = query_interface.inspect_concept(concept)
delta = query_interface.diff(concept, antonym)

print('Concept inspection:', concept)
print(info)
print('
Diff against antonym:', antonym)
print(delta)
